In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.decomposition import PCA

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

from sklearn.metrics import r2_score,root_mean_squared_error, mean_squared_error, mean_absolute_error
from sklearn.model_selection import LeaveOneGroupOut

In [51]:
#File checker
DATA_DIR = Path("data")
data_files = sorted(
    p for p in DATA_DIR.iterdir()
    if p.suffix.lower() in {".csv"}#, ".parquet", ".pkl", ".pickle"}
)
df_raw = pd.read_csv(
    DATA_DIR / "perera_pfizer_raw_download.csv"
)
# df_raw = pd.read_csv(
#     PERERA_URL,
#     sep=";",
#     decimal=",",
#     encoding="utf-8-sig"
# )
print("Directory found:", DATA_DIR)
print("scikitlearn version:", sklearn.__version__)
print("Files found:")
for p in data_files:
    print(p.name)
    print(pd.read_csv(p, sep=",", decimal=",", encoding="utf-8-sig").shape)
    print(pd.read_csv(p, sep=",", decimal=",", encoding="utf-8-sig").head(1).columns.tolist())

Directory found: data
scikitlearn version: 1.9.0
Files found:
kraken_features_only.csv
(1223, 191)
['id', 'vmin_vmin_boltz', 'vmin_r_boltz', 'fmo_e_homo_boltz', 'fmo_e_lumo_boltz', 'fmo_mu_boltz', 'fmo_eta_boltz', 'fmo_omega_boltz', 'somo_ra_boltz', 'somo_rc_boltz', 'nbo_P_boltz', 'nbo_P_ra_boltz', 'spindens_P_ra_boltz', 'nbo_P_rc_boltz', 'spindens_P_rc_boltz', 'nmr_P_boltz', 'nmrtens_sxx_P_boltz', 'nmrtens_syy_P_boltz', 'nmrtens_szz_P_boltz', 'efg_amp_P_boltz', 'efgtens_xx_P_boltz', 'efgtens_yy_P_boltz', 'efgtens_zz_P_boltz', 'nuesp_P_boltz', 'E_solv_cds_boltz', 'nbo_lp_P_percent_s_boltz', 'nbo_lp_P_occ_boltz', 'nbo_lp_P_e_boltz', 'nbo_bd_e_max_boltz', 'nbo_bd_e_avg_boltz', 'nbo_bds_e_min_boltz', 'nbo_bds_e_avg_boltz', 'nbo_bd_occ_min_boltz', 'nbo_bd_occ_avg_boltz', 'nbo_bds_occ_max_boltz', 'nbo_bds_occ_avg_boltz', 'E_solv_total_boltz', 'E_solv_elstat_boltz', 'E_oxidation_boltz', 'E_reduction_boltz', 'fukui_p_boltz', 'fukui_m_boltz', 'vbur_vtot_boltz', 'vbur_ratio_vbur_vtot_boltz', 'P

In [65]:
#Load in data sets
df_interp= pd.read_csv(DATA_DIR /"perera_pfizer_group1_kraken_interpretable.csv")
df_full = pd.read_csv(DATA_DIR / "perera_pfizer_group1_kraken_full.csv")

#drop static  features
df_interp = df_interp.loc[:, df_interp.nunique(dropna=False) > 1].copy()
df_full = df_full.loc[:, df_full.nunique(dropna=False) > 1].copy()
 
#Target variable
TARGET = "Product_Yield_PCT_Area_UV"

#Non catalyst variables
non_catalyst = ["Reactant_2_Name", "Reactant_1_Name", "Solvent_1_Short_Hand", "ligand"]

#Drop additional yield feature
df_interp = df_interp.drop(columns=["Product_Yield_Mass_Ion_Count"])
df_full = df_full.drop(columns=["Product_Yield_Mass_Ion_Count"])
#ligand one hot encoding
df_onehot_ligand =df_interp.copy().drop(columns=["%vbur_boltz","%vbur_min","Reaction_No","Reactant_1_Short_Hand","Reagent_combo","reactant1_code","reactant2_code","Reactant_1_Name","Reactant_2_Name"])
#Drop non needed reaction based features
df_interp = df_interp.drop(columns=["Reaction_No","Reactant_1_Short_Hand","Reagent_combo","reactant1_code","reactant2_code","kraken_id","Reactant_1_Name","Reactant_2_Name"])#,"Reagent_1_Short_Hand",,"substrate_pair",])
df_full = df_full.drop(columns=["Reaction_No","Reactant_1_Short_Hand","Reagent_combo","reactant1_code","reactant2_code","Reactant_1_Name","Reactant_2_Name","kraken_id"])#,"Reagent_1_Short_Hand",,"substrate_pair",])
#Drop catalyst features in interpretable set to only the 5 permitted
df_five_interp = df_interp.copy().drop(columns=["vmin_vmin_boltz","fmo_e_homo_boltz","fmo_e_lumo_boltz","%vbur_max"])
#reduce to only 2 catalyst features
df_bur_boltz_vbur_min = df_five_interp.copy().drop(columns=["dipolemoment_boltz","%vbur_delta", "delta_band_gap"])


In [ ]:
for col in df_interp.columns:
    print(f"\n--- {col} ---")
    print(df_interp[col].value_counts(dropna=False))


--- ligand ---
ligand
P(tBu)3        384
P(Ph)3         384
AmPhos         384
P(Cy)3         384
P(o-Tol)3      384
CataCXium A    384
SPhos          384
XPhos          384
Name: count, dtype: int64

--- Reagent_1_Short_Hand ---
Reagent_1_Short_Hand
NaOH      384
NaHCO3    384
CsF       384
K3PO4     384
KOH       384
LiOtBu    384
Et3N      384
NaN       384
Name: count, dtype: int64

--- Solvent_1_Short_Hand ---
Solvent_1_Short_Hand
MeCN    768
THF     768
DMF     768
MeOH    768
Name: count, dtype: int64

--- Product_Yield_PCT_Area_UV ---
Product_Yield_PCT_Area_UV
70.45    5
13.04    5
15.05    5
17.77    5
13.53    5
        ..
18.14    1
17.83    1
20.89    1
48.66    1
43.45    1
Name: count, Length: 2510, dtype: int64

--- substrate_pair ---
substrate_pair
1a_2a    256
1b_2a    256
1c_2a    256
1d_2a    256
1c_2b    256
1d_2b    256
1b_2b    256
1a_2b    256
1c_2c    256
1d_2c    256
1b_2c    256
1a_2c    256
Name: count, dtype: int64

--- vmin_vmin_boltz ---
vmin_vmin_boltz
-

PCA

In [66]:
# ---- Fit PCA on the full kraken descriptor table and expose the loadings ----
# kraken_features_only.csv = one row per ligand ("id") + the 190 physical-organic descriptors.
# Per the SI, descriptors are standard-scaled BEFORE PCA.

df_desc = pd.read_csv(DATA_DIR / "kraken_features_only.csv")

# Descriptor columns = everything except the ligand identifier
DESC_ID_COL = "id"
desc_cols = [c for c in df_desc.columns if c != DESC_ID_COL]

# Drop any descriptor that is entirely NaN, then fill remaining gaps with column means
X_desc = df_desc[desc_cols].apply(pd.to_numeric, errors="coerce")
X_desc = X_desc.dropna(axis=1, how="all")
desc_cols = X_desc.columns.tolist()
X_desc = X_desc.fillna(X_desc.mean())

N_COMPONENTS = 10  # keep the first 10 PCs (paper reports variance for PC1..PC10)

pca_pipe = Pipeline([
    ("scale", StandardScaler()),
    ("pca", PCA(n_components=N_COMPONENTS, random_state=42)),
])
scores = pca_pipe.fit_transform(X_desc)                 # per-ligand PC scores
pca = pca_pipe.named_steps["pca"]

pc_names = [f"PC{i+1}" for i in range(N_COMPONENTS)]

# Loadings: rows = original descriptors, cols = PCs. (components_ is [n_pc, n_features])
loadings = pd.DataFrame(pca.components_.T, index=desc_cols, columns=pc_names)

# Explained variance (sanity check against Table S7: ~28.0, 12.9, 11.2, 6.6 ...)
explained = pd.Series(pca.explained_variance_ratio_, index=pc_names, name="explained_var_ratio")
print("Explained variance ratio:")
print((explained * 100).round(1).to_string())
loadings.head()


Explained variance ratio:
PC1     29.0
PC2     13.0
PC3     11.2
PC4      6.1
PC5      4.8
PC6      4.5
PC7      3.5
PC8      2.5
PC9      2.3
PC10     1.9


,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10
vmin_vmin_boltz,-0.040673,0.109820,-0.041338,0.131300,0.016202,0.037612,0.070836,-0.161692,-0.050804,0.021805
vmin_r_boltz,-0.031977,0.094786,-0.029553,0.106694,0.001846,-0.010934,0.078246,-0.204389,-0.051469,0.006858
fmo_e_homo_boltz,0.068985,-0.080719,0.007620,-0.067753,-0.037611,0.016744,-0.103893,0.172853,0.105747,0.105483
fmo_e_lumo_boltz,-0.051804,-0.065353,0.107962,-0.040589,-0.043974,0.065513,-0.012614,0.166571,0.084007,-0.103619
fmo_mu_boltz,0.000139,-0.092731,0.083557,-0.067352,-0.053145,0.057416,-0.066810,0.218140,0.120301,-0.017723


In [67]:
# ---- Rank feature contributions for any PC ----
def rank_loadings(pc, n=None, by_abs=True):
    """Return descriptors ranked by their loading on `pc` (e.g. 'PC1')."""
    s = loadings[pc]
    order = s.abs().sort_values(ascending=False) if by_abs else s.sort_values(ascending=False)
    out = pd.DataFrame({"loading": s.loc[order.index], "abs_loading": s.loc[order.index].abs()})
    return out.head(n) if n else out

# Example: top 10 contributors to PC1
rank_loadings("PC1", 10)


,loading,abs_loading
vbur_vtot_boltz,0.124170,0.124170
volume_boltz,0.123924,0.123924
vbur_near_vtot_vburminconf,0.122562,0.122562
vbur_near_vtot_max,0.122505,0.122505
surface_area_boltz,0.122362,0.122362
Pint_P_max_boltz,0.119212,0.119212
Pint_P_int_boltz,0.117934,0.117934
vbur_qvtot_max_max,0.116291,0.116291
vbur_qvtot_max_boltz,0.115987,0.115987
vbur_qvtot_max_vburminconf,0.115516,0.115516


In [68]:
# ---- Top-3 descriptors from each of PC1..PC4 -> merge onto the interpretable / one-hot table ----
TOP_K = 3
TOP_PCS = ["PC1", "PC2", "PC3", "PC4"]

top_desc_per_pc = {pc: rank_loadings(pc, TOP_K).index.tolist() for pc in TOP_PCS}
for pc, cols in top_desc_per_pc.items():
    print(pc, "->", cols)

# De-duplicated, order-preserving list of the raw descriptor columns to add
pc_top_features = list(dict.fromkeys(c for cols in top_desc_per_pc.values() for c in cols))
print("\nDescriptors added from PCs:", pc_top_features)

# Per-ligand lookup table of just those descriptors, keyed by kraken id
desc_lookup = df_desc[[DESC_ID_COL] + pc_top_features].copy()

# Choose the base table to augment. Prefer df_onehot_ligand if you've already built it;
# otherwise fall back to df_interp. Both are keyed to a ligand via kraken id.
base_df = df_onehot_ligand.copy() if "df_onehot_ligand" in dir() else df_interp.copy()

# Figure out which column in base_df holds the kraken ligand id.
id_candidates = [c for c in ("kraken_id", "id", "ligand") if c in base_df.columns]
if not id_candidates:
    raise KeyError("No kraken id column ('kraken_id'/'id'/'ligand') found in base_df to merge on.")
BASE_ID_COL = id_candidates[0]
print("id", id_candidates)
df_pca_augmented = base_df.merge(
    desc_lookup, how="left", left_on=BASE_ID_COL, right_on=DESC_ID_COL,
)
if DESC_ID_COL != BASE_ID_COL and DESC_ID_COL in df_pca_augmented.columns:
    df_pca_augmented = df_pca_augmented.drop(columns=[DESC_ID_COL])

missing = df_pca_augmented[pc_top_features].isna().any(axis=1).sum()
print(f"\nMerged on '{BASE_ID_COL}'. Rows with unmatched descriptors: {missing}")
print("df_pca_augmented shape:", df_pca_augmented.shape)
df_pca_augmented.head()


PC1 -> ['vbur_vtot_boltz', 'volume_boltz', 'vbur_near_vtot_vburminconf']
PC2 -> ['pyr_P_max', 'qpole_amp_max', 'vbur_qvbur_min_min']
PC3 -> ['vbur_near_vbur_delta', 'vbur_vbur_delta', 'vbur_qvbur_min_delta']
PC4 -> ['nbo_bds_occ_avg_boltz', 'efgtens_zz_P_boltz', 'nmr_P_boltz']

Descriptors added from PCs: ['vbur_vtot_boltz', 'volume_boltz', 'vbur_near_vtot_vburminconf', 'pyr_P_max', 'qpole_amp_max', 'vbur_qvbur_min_min', 'vbur_near_vbur_delta', 'vbur_vbur_delta', 'vbur_qvbur_min_delta', 'nbo_bds_occ_avg_boltz', 'efgtens_zz_P_boltz', 'nmr_P_boltz']
id ['kraken_id', 'ligand']

Merged on 'kraken_id'. Rows with unmatched descriptors: 0
df_pca_augmented shape: (3072, 25)


,ligand,Reagent_1_Short_Hand,Solvent_1_Short_Hand,Product_Yield_PCT_Area_UV,substrate_pair,kraken_id,vmin_vmin_boltz,fmo_e_homo_boltz,fmo_e_lumo_boltz,dipolemoment_boltz,...,vbur_near_vtot_vburminconf,pyr_P_max,qpole_amp_max,vbur_qvbur_min_min,vbur_near_vbur_delta,vbur_vbur_delta,vbur_qvbur_min_delta,nbo_bds_occ_avg_boltz,efgtens_zz_P_boltz,nmr_P_boltz
0,P(tBu)3,NaOH,MeCN,4.76,1a_2a,8,-0.067185,-5.901605,0.979882,1.083268,...,253.200425,0.830704,2.814976,15.199086,0.000000,0.000000,0.000000,0.052453,1.428239,225.189000
1,P(Ph)3,NaOH,MeCN,4.12,1a_2a,17,-0.048230,-6.214809,-0.802464,1.460511,...,302.715907,0.932082,4.561356,11.489263,0.000000,0.000000,0.000000,0.037103,1.456498,296.152700
2,AmPhos,NaOH,MeCN,2.58,1a_2a,216,-0.070188,-5.596838,-0.233746,3.240919,...,316.760137,0.855551,12.311883,13.548649,0.000000,0.000000,0.000000,0.050737,1.432126,256.655600
3,P(Cy)3,NaOH,MeCN,4.44,1a_2a,11,-0.066835,-5.990187,0.872594,1.280457,...,325.712062,0.937360,4.044921,9.958058,12.226625,17.283574,3.696227,0.032226,1.372408,283.164838
4,P(o-Tol)3,NaOH,MeCN,1.95,1a_2a,9,-0.043479,-6.056948,-0.764621,0.432101,...,349.914566,0.918251,6.600232,11.333423,8.174791,10.024996,4.617669,0.041685,1.456571,317.294407


In [41]:
# ---- (Optional) per-ligand PC SCORES, if you'd rather model on PCs directly than raw descriptors ----
df_pc_scores = pd.DataFrame(scores[:, :N_COMPONENTS], columns=pc_names)
df_pc_scores.insert(0, DESC_ID_COL, df_desc[DESC_ID_COL].values)

# Attach the first few PC scores to the augmented table for later GP use
KEEP_PC_SCORES = ["PC1", "PC2", "PC3", "PC4"]
df_pca_augmented = df_pca_augmented.merge(
    df_pc_scores[[DESC_ID_COL] + KEEP_PC_SCORES],
    how="left", left_on=BASE_ID_COL, right_on=DESC_ID_COL,
    suffixes=("", "_score"),
)
if DESC_ID_COL != BASE_ID_COL and DESC_ID_COL in df_pca_augmented.columns:
    df_pca_augmented = df_pca_augmented.drop(columns=[DESC_ID_COL])

df_pca_augmented.head()


NameError: name 'df_pca_augmented' is not defined